# v8 GQA M-packing — the gate (Cut 1: CUDA-core, vast.ai T4)

v8 packs the `G = H_q/H_kv` query heads that share one KV head into the score GEMM's **M** dimension: a CTA reads that KV head **once** and runs **G query rows** against it (G warps active, not 1), so decode `AI = 2/b -> 2G/b`. This is Cut 1 — CUDA-core, sm_75/T4 — isolating the M-packing variable cheaply. **Gate 1 = correctness (v8 + v7 regression); Gate 2 = the quiz.**

## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    import torch  # already on the image? keep it, don't churn the version
except ImportError:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
pip('ninja', 'pytest', 'numpy')

# vast.ai venv fix: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5
/usr/local/bin/python
shell python sees torch 2.11.0+cu128


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Roofline first — decode `AI = 2G/b` (the per-step gate, recorded BEFORE the bench)

Predict on **sm_80 (A100, the Cut-2 perf target)**: AI rises `G×`, the HBM floor drops `G×`, but the limiter **stays HBM** (A100 fp16 ridge = 153, so even G=8's AI=8 is far below). The win is per-CTA efficiency, not a limiter flip — that's what the bench must show.

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_80')   # AI is arch-independent; the t_hbm floor is the A100 target's
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'ridge':>6} | {'t_hbm floor':>12}")
for G in (1, 2, 4, 8, 16, 32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.ridge:6.1f} | {e.t_hbm*1e3:9.4f}ms')
print()
print('Prediction: AI rises Gx, floor drops Gx, limiter STAYS HBM (AI < ridge 153 even at G=8).')
print('v8 win = per-CTA efficiency (G warps + KV read once), NOT a limiter flip. Measured below.')


  G |  AI=2G/b | limiter |  ridge |  t_hbm floor
  1 |      1.0 |     HBM |  153.0 |    0.1317ms
  2 |      2.0 |     HBM |  153.0 |    0.0658ms
  4 |      4.0 |     HBM |  153.0 |    0.0329ms
  8 |      8.0 |     HBM |  153.0 |    0.0165ms
 16 |     16.0 |     HBM |  153.0 |    0.0082ms
 32 |     31.9 |     HBM |  153.0 |    0.0041ms

Prediction: AI rises Gx, floor drops Gx, limiter STAYS HBM (AI < ridge 153 even at G=8).
v8 win = per-CTA efficiency (G warps + KV read once), NOT a limiter flip. Measured below.


## 3. Build v8 (JIT)

In [4]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v8_gqa')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v8_gqa')
print('built:', mod)


built: <module 'fa_v8_gqa' from '/root/.cache/torch_extensions/py312_cu128/fa_v8_gqa/fa_v8_gqa.so'>


## 4. Correctness gate — v8 GQA + v7 regression  *(Gate 1 of 2)*

v8's GQA cases (decode `G in {1,2,4,8}`, non-multiple `N_k`, causal+offset, idle-warp `G=3` + multi-tile `G=16`, square reduction) **and** v7 unchanged. Oracle = `sdpa_reference_gqa` (KV expanded by `repeat_interleave(G)`).

In [5]:
!python -m pytest tests/test_correctness.py -k "v8_gqa or v7_paged" -q


................................................................         [100%]
64 passed, 100 deselected in 81.81s (0:01:21)


## 5. Decode benchmark — GQA `G=8` canonical (non-causal + causal)

`--heads 8 --gqa-group 8` -> H_q=8 query heads, H_kv=1 KV head. `vs naive` here is **v7 on the same attention with KV expanded to H_q heads** (the no-M-packing floor) — the clean same-session isolation of M-packing's one variable. `vs sdpa` is the torch GQA baseline.

In [6]:
!python -m bench.harness --backend v8_gqa --decode --heads 8 --gqa-group 8
print()
!python -m bench.harness --backend v8_gqa --decode --heads 8 --gqa-group 8 --causal


# device: Tesla T4 (sm_75)  clock~360/1590MHz  backend=v8_gqa  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
ninja: no work to do.
     1x8x1x64/2048 G8 |   0.235/  0.290 |    29.34 |   0.7% |    3.22x |    1.26x | HBM (~0.00ms)
    1x8x1x128/2048 G8 |   0.290/  0.347 |    36.23 |   1.1% |    2.42x |    1.29x | HBM (~0.00ms)
     1x8x1x64/8192 G8 |   0.187/  0.203 |    23.34 |   3.5% |   11.25x |    3.44x | HBM (~0.01ms)
    1x8x1x128/8192 G8 |   0.239/  0.261 |    29.85 |   5.5% |    7.68x |    3.87x | HBM (~0.01ms)
    1x8x1x64/16384 G8 |   0.301/  0.321 |    37.59 |   4.4% |   11.24x |    3.62x | HBM (~0.01ms)
   1x8x1x128/16384 G8 |   0.432/  0.494 |    54.00 |   6.1% |    8.40x |    4.25x | HBM (~0.03ms)

# device: Tesla T4 (sm_75)  clock~360/1590MHz  backend=v8_gqa  precision=fp32  causal=True  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM

## 6. The G-sweep — GEMV->GEMM per-CTA efficiency (the v8 deliverable)

`--heads 32` fixes H_q=32; `H_kv = 32//G` shrinks as G grows (KV bytes drop by G). Watch `us/tok` fall and `vs naive` (= vs v7 no-packing) rise as G activates G warps + reads KV once. The prediction-vs-measured curve as G crosses the M<16->M>=16 threshold (the tensor-core line is Cut 2 / A100).

In [7]:
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


# device: Tesla T4 (sm_75)  clock~390/1590MHz  backend=v8_gqa  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
ninja: no work to do.
    1x32x1x64/8192 G1 |   3.281/  4.819 |   102.52 |   6.4% |    0.55x |    0.85x | HBM (~0.21ms)
    1x32x1x64/8192 G2 |   1.122/  1.189 |    35.07 |   9.3% |    2.61x |    2.52x | HBM (~0.10ms)
    1x32x1x64/8192 G4 |   0.586/  0.625 |    18.31 |   8.9% |    4.99x |    4.79x | HBM (~0.05ms)
    1x32x1x64/8192 G8 |   0.327/  0.363 |    10.21 |   8.0% |    8.92x |    8.59x | HBM (~0.03ms)
   1x32x1x64/8192 G16 |   0.270/  0.345 |     8.44 |   4.9% |   10.76x |   10.42x | HBM (~0.01ms)
   1x32x1x64/8192 G32 |   0.221/  0.385 |     6.92 |   3.0% |   13.05x |   12.74x | HBM (~0.01ms)
   1x32x1x128/8192 G1 |   4.545/  4.694 |   142.02 |   9.2% |    0.43x |    1.00x | HBM (~0.42ms)
   1x32x1x128/8192 G2 |   1.880/  1.925 |    58.74 |  11.2% |    2.21

## 7. Reclaim SDPA at batch — the headline the v7 data created

v7 **lost** to torch SDPA at B>=8 (0.5x). With G=8 (8 warps active, KV read once) v8 must reclaim the serving-batch regime. Hold G=8, sweep B; `vs sdpa` is the column that matters.

In [8]:
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64


# device: Tesla T4 (sm_75)  clock~360/1590MHz  backend=v8_gqa  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
ninja: no work to do.
     1x8x1x64/8192 G8 |   0.370/  0.422 |    46.22 |   1.8% |    9.93x |    1.72x | HBM (~0.01ms)
    1x8x1x128/8192 G8 |   0.268/  0.333 |    33.52 |   4.9% |    7.81x |    3.46x | HBM (~0.01ms)
     8x8x1x64/8192 G8 |   0.633/  0.674 |     9.89 |   8.3% |    8.40x |    7.67x | HBM (~0.05ms)
    8x8x1x128/8192 G8 |   1.046/  1.085 |    16.34 |  10.0% |    7.62x |    7.66x | HBM (~0.10ms)
    16x8x1x64/8192 G8 |   1.290/  1.313 |    10.08 |   8.1% |    8.21x |    7.44x | HBM (~0.10ms)
   16x8x1x128/8192 G8 |   2.159/  2.192 |    16.86 |   9.7% |    7.35x |    7.43x | HBM (~0.21ms)
    32x8x1x64/8192 G8 |   3.165/  3.282 |    12.36 |   6.6% |    6.38x |    6.08x | HBM (~0.21ms)
   32x8x1x128/8192 G8 |   5.080/  5.156 |    19.84 |   8.3% |    6.09